# SpaNCy-Shift — graph-fix A/B (old graph vs fixed graph)

Copy of `spancy_shift_ablation.ipynb`. It compares the Stage 2 GNN trained with the **original
graph code** (`spancy_shift.py`) against the **fixed graph code** (`spancy_shift_graphfix.py`):

1. **Edge direction**: edges now point neighbour → cell, so each cell aggregates its own k spatial neighbours.
2. **Dense training graph**: `PatchSampler` draws spatially contiguous patches instead of scattered cells.
3. **2-hop halo**: training and inference both see every cell's full receptive field, so the output no longer depends on `inference_batch_size`.

**To run:** Runtime → Run all. Upload **`spancy_shift.py`**, **`spancy_shift_graphfix.py`** and **`mmd_resnet_baseline.py`** when §0 asks.
§4d adds **MMD-ResNet** (Shaham et al. 2017) as an external baseline arm.
Paste back the outputs of **§4c** (sanity checks), **§8** (kBET + silhouette) and **§9** (seed repeat).

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads `spancy_shift.py`.

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload BOTH module files: the original (control arm + shared Stage 1/diagnostics)
# and the graph-fix copy (treatment arm). Only missing files are requested.
from google.colab import files
import os

NEEDED = ['spancy_shift.py', 'spancy_shift_graphfix.py', 'mmd_resnet_baseline.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:28s} {os.path.getsize(f):>10,} bytes')

In [ ]:
import gc
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import anndata as ad
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

# Control arm: original graph code. Stage 1 and all diagnostics are identical in both
# modules, so the shared helpers are imported from the original.
import spancy_shift
from spancy_shift import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    shift_normalize_per_marker,
    GNNStage2, AdjacencyIndex, train,
    normalize_adata,
    positive_population_table, summarize_positive_population,
    per_marker_batch_r2,
)

# Treatment arm: fixed graph code (edge direction, patch sampler, 2-hop halo).
import spancy_shift_graphfix as gf
# External baseline: MMD-ResNet (Shaham et al. 2017), ported from the authors' code.
import mmd_resnet_baseline as mrn

# After re-uploading any file:
# importlib.reload(spancy_shift); importlib.reload(gf); importlib.reload(mrn)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)

In [ ]:
# Per-batch mean intensity overview
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].boxplot([X_raw[:, i] for i in range(X_raw.shape[1])], labels=marker_names, vert=True)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')

for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Bimodal Marker Detection (preview on raw data)

Adaptive per-batch voting: a marker is bimodal if ≥50% of batches independently show ≥2
prominent histogram peaks. Used in Stage 1 (log1p space).

- **Bimodal**: ≥2 peaks in majority of batches → separate neg/pos peak shifts per sample
- **Unimodal**: 1 peak → single median shift per sample

This cell previews bimodal detection on raw data. The actual detection runs inside
`shift_normalize_per_marker()` (log1p space) and the result is passed directly into Stage 2
— Stage 2 no longer re-detects bimodal markers on scaled data.

In [ ]:
# Preview: scale raw data for bimodal detection visualization
# Note: train() will run bimodal detection on log1p(X_base_scaled) internally
X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
print(f'log1p range: [{X_log_preview.min():.3f}, {X_log_preview.max():.3f}]')

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

print(f"\nBimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")

print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")

## 2b. Reference Sample Selection

For each marker, find the **medoid sample** — the one with lowest mean pairwise symmetric KL
divergence to all other samples in log1p space.  This sample is the normalization target for
that marker: all other samples are pulled toward it.  Using a data-driven reference (rather
than the global mean/quantile) is the key improvement over the previous spancy_shift approach.

In [ ]:
# Compute reference sample per marker (passed to train() to skip recomputation)
ref_sample_per_marker = find_best_sample_per_marker(adata)

# Show as table
ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample (normalization target):\n')
print(ref_df.to_string(index=False))

# How many distinct reference samples are used?
ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples used: {len(ref_counts)}')
print(ref_counts.to_string())

# Heatmap: for each marker, which sample is the reference?
fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = np.zeros((1, len(sample_order)))
for i, s in enumerate(sample_order):
    ref_matrix[0, i] = sum(v == s for v in ref_sample_per_marker.values())
ax.bar(range(len(sample_order)), ref_matrix[0], color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('How often each sample is chosen as reference (medoid across markers)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Stage 2 Training (GNN)

Trains `GNNStage2` on Stage 1 output (`X_base`).

**Inside `train()`:**
1. Runs `shift_normalize_per_marker()` → `X_base` (Stage 1, analytic)
2. Fits `RobustScaler` on `log1p(X_base)` → `X_base_scaled`
3. Builds spatial k-NN graph from (x, y) coordinates per scene
4. Trains `GNNStage2` with:
   - `L_recon` — Huber (keep delta small, biology-preserving)
   - `L_contrast` — NT-Xent with spatial neighbors as positives
   - `L_adv` — adversarial batch discriminator with gradient reversal (GRL)

**Inference:** `X_out = X_base_scaled + hybrid_alpha * delta` → inverse_transform

**Key parameters:**
- `hybrid_alpha=0.6` — blend strength: 0=Stage 1 only, 1=full GNN residual (0.6 = operating point)
- `n_per_batch=512` — cells per batch per step in scene-based sampler
- `k_neighbors=15` — spatial k-NN connectivity
- `w_contrast=0.5` — spatial NT-Xent weight
- `w_adv=0.3` — adversarial loss weight (GRL lambda ramps 0→1 over training)

In [ ]:
HYBRID_ALPHA = 0.6
K_NEIGHBORS = 15
N_EPOCHS = 10  # same as the published operating point

# ── A/B CONTROL ARM: OLD GRAPH ───────────────────────────────────────────────
# spancy_shift.train = the ORIGINAL graph code, at the current default loss
# (w_adv=0.0, w_contrast=0.5). This is the 0.743-kBET configuration of the
# 2026-09-28 loss ablation. §4b trains the graph-fix arms from the same seed with
# identical hyperparameters, so the only difference is the graph code.
ABL_SEED = 0
torch.manual_seed(ABL_SEED)
torch.cuda.manual_seed_all(ABL_SEED)

model, trained_scaler, trained_ref, history = spancy_shift.train(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    device_str=DEVICE,
    warmup_epochs=2,
    n_per_batch=512,
    k_neighbors=K_NEIGHBORS,
    hybrid_alpha=HYBRID_ALPHA,
    temperature=0.07,
    w_recon=0.1,    # soft regularizer only — MMD is the primary decoder signal
    w_contrast=0.5,
    w_adv=0.0,      # current default (adversarial CE off)
    w_mmd=1.0,      # MMD on decoder output (unimodal markers only, across batches)
    mmd_samples=256,
    grl_max=1.0,
    bimodal_min_batch_frac=0.5,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete (old graph).')
print(f'  Final loss={history["loss"][-1]:.4f}  '
      f'recon={history["recon"][-1]:.4f}  '
      f'contrast={history["contrast"][-1]:.4f}  '
      f'mmd={history["mmd"][-1]:.4f}')

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 6, figsize=(28, 4))

for ax, key, color, title in zip(
    axes,
    ['loss', 'recon', 'contrast', 'adv', 'mmd', 'lr'],
    ['k', 'darkorange', 'steelblue', 'firebrick', 'mediumorchid', 'purple'],
    ['Total Loss', 'Recon (Huber)', 'Contrast (NT-Xent)', 'Adversarial (GRL)', 'MMD (decoder out)', 'Learning Rate'],
):
    ax.plot(history[key], color=color, linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    if key == 'lr':
        ax.set_yscale('log')

ax_grl = axes[3].twinx()
ax_grl.plot(history['grl_lambda'], 'g--', linewidth=1, alpha=0.6)
ax_grl.set_ylabel('GRL λ', color='green', fontsize=9)

plt.suptitle('Stage 2 Training — GNNStage2', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Inference & Output Inspection

Run inference with both affine-only and shift modes.
Use `shift_alpha` to blend between affine (0) and full shift (1).

In [ ]:
# Run both stages: Stage 1 (analytic) + Stage 2 (GNN residual)
# normalize_adata() runs Stage 1 internally and stores both layers.

adata_norm = normalize_adata(
    adata, model, trained_scaler, trained_ref,
    hybrid_alpha=HYBRID_ALPHA,
    k_neighbors=K_NEIGHBORS,
    inference_batch_size=2048,  # reduce if OOM; increase if GPU has headroom
    device_str=DEVICE,
    layer_name='normalized',
    keep_base_layer=True,
)

# Convenience arrays
X_base = np.asarray(adata_norm.layers['normalized_base'])   # Stage 1
X_norm = np.asarray(adata_norm.layers['normalized'])         # Stage 2

print(f'Stage 1 (normalized_base): min={X_base.min():.4f}  max={X_base.max():.4f}  mean={X_base.mean():.4f}')
print(f'Stage 2 (normalized):      min={X_norm.min():.4f}  max={X_norm.max():.4f}  mean={X_norm.mean():.4f}')
print(f'\nLayers: {list(adata_norm.layers.keys())}')

In [ ]:
# Delta magnitude: how much the GNN is correcting each marker
n_viz = min(5000, adata_norm.n_obs)
model.eval()
with torch.no_grad():
    X_base_viz = np.asarray(adata_norm.layers['normalized_base'][:n_viz], dtype=np.float32)
    X_sc_viz = torch.tensor(
        trained_scaler.transform(np.log1p(np.clip(X_base_viz, 0, None))).astype(np.float32)
    ).to(DEVICE)
    edge_dummy = torch.zeros((2, 0), dtype=torch.long, device=DEVICE)
    _, delta_viz, _, _ = model(X_sc_viz, edge_dummy)
    delta_np = delta_viz.cpu().numpy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot([delta_np[:, m] for m in range(len(marker_names))], labels=marker_names, vert=True)
ax.set_xticklabels(marker_names, rotation=90, fontsize=8)
ax.set_ylabel('Delta (scaled space)')
ax.set_title(f'GNN Residual Delta per Marker (first {n_viz} cells, no spatial context)')
ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Compare raw vs Stage 1 vs Stage 2 for unimodal markers
# Bimodal markers are masked from Stage 2 MMD and pass through unchanged from Stage 1 —
# their histograms will always match Stage 1 exactly and prove nothing about Stage 2 quality.
# Use unimodal markers where Stage 2 actually makes corrections.
unimodal_names = [marker_names[i] for i in range(len(marker_names)) if not marker_is_bimodal[i]]
plot_markers = unimodal_names[:4] if len(unimodal_names) >= 4 else unimodal_names

layers_to_show = [
    (None,                 'Raw'),
    ('normalized_base',    'Stage 1 (analytic)'),
    ('normalized',         'Stage 2 (DL + MMD)'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        if layer_key is None:
            data = X_raw
        else:
            data = adata_norm.layers[layer_key]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(np.asarray(data)[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle('Unimodal Marker Distributions: Raw → Stage 1 → Stage 2\n'
             '(unimodal markers chosen — bimodal markers are masked in Stage 2 MMD and unchanged)', fontsize=12)
plt.tight_layout()
plt.show()

## 4b. Graph-fix A/B — train the fixed-graph arms

The control arm (`normalized`, **old-graph**) was trained in §3 with `spancy_shift.train`.
The arms below use `spancy_shift_graphfix` with the **same seed and hyperparameters**, so they
differ only in the graph code.

| Arm | Module | `w_contrast` | `w_adv` | Layer |
|---|---|---|---|---|
| old-graph (control) | `spancy_shift` | 0.5 | 0.0 | `normalized` (§3) |
| **graph-fix** | `spancy_shift_graphfix` | 0.5 | 0.0 | `norm_fix` |
| graph-fix MMD-only | `spancy_shift_graphfix` | **0.0** | 0.0 | `norm_fix_mmdonly` |
| **no-graph** | `spancy_shift_graphfix`, `graph_mode='none'` | 0.5 | 0.0 | `norm_nograph` |

- `graph-fix MMD-only` asks whether NT-Xent matters once it has real spatial positives. With the old
  sampler it had almost none.
- `no-graph` is the **per-cell control**: the same model with an empty edge list, so the GAT does no
  message passing. It uses the original sampler with the same seed, so its mini-batches and NT-Xent
  pairs match old-graph exactly. If no-graph ≈ old-graph, the published result never needed the
  graph: the old sampler gave each cell only ~0.3 of 15 neighbours.

Every downstream section (§5 adj-R², §6 positive population + shape, §7 histograms, §8 kBET,
silhouette) reports all five layers (Stage 1 + four arms) at α=0.6.

### Decision rule (fixed in advance)
- **graph-fix ≥ old-graph on kBET, biology no worse** (silhouette, shape, pos-pop) → adopt; merge
  the fix into `spancy_shift.py`.
- **graph-fix worse on kBET** → a truly spatial model is not buying anything. The old "GNN" was
  effectively a per-cell model; report that, and consider re-tuning `w_contrast` / α.
- **Within seed noise** (see §9) → keep the old graph; report only the §4c check results.
- **no-graph ≈ old-graph** (kBET, shape, pos-pop) → the published model is a per-cell residual
  corrector in practice; describe it that way (no spatial claim).

In [ ]:
# ============================================================================
# 4b. GRAPH-FIX A/B — train the fixed-graph arms
# ----------------------------------------------------------------------------
# The control arm (old graph) is already trained (§3) and lives in
# adata_norm.layers['normalized']. Each arm here starts from the same seed.
# Stage 1 is deterministic and identical in both modules, so every arm sits on a
# byte-identical X_base and 'normalized_base' never changes.
# ============================================================================

ABL_ARMS = {
    # layer_key          : (label,                module, w_contrast, w_adv, extra kwargs)
    'norm_fix':           ('graph-fix',          gf,     0.5,        0.0,   {}),
    'norm_fix_mmdonly':   ('graph-fix MMD-only', gf,     0.0,        0.0,   {}),
    # per-cell control: empty edge list, ORIGINAL sampler (same batches as old-graph)
    'norm_nograph':       ('no-graph',           gf,     0.5,        0.0,   {'graph_mode': 'none'}),
}

# Canonical layer list consumed by every downstream section.
ABL_LAYERS = [('normalized_base', 'Stage 1'), ('normalized', 'old-graph')] + \
             [(k, v[0]) for k, v in ABL_ARMS.items()]

abl_history = {'normalized': history}   # control arm, from §3
fix_model = fix_scaler = None

for layer_key, (label, mod, w_contrast, w_adv, kw) in ABL_ARMS.items():
    print('\n' + '=' * 70)
    print(f'ARM: {label:>18s}   module={mod.__name__}  w_contrast={w_contrast}  '
          f'w_adv={w_adv}  {kw}  -> {layer_key}')
    print('=' * 70)

    # identical init to the §3 control arm — only the graph code (and w_contrast) differ
    torch.manual_seed(ABL_SEED)
    torch.cuda.manual_seed_all(ABL_SEED)

    m_arm, sc_arm, ref_arm, h_arm = mod.train(
        adata,
        n_epochs=N_EPOCHS,
        lr=1e-3,
        device_str=DEVICE,
        warmup_epochs=2,
        n_per_batch=512,
        k_neighbors=K_NEIGHBORS,
        hybrid_alpha=HYBRID_ALPHA,
        temperature=0.07,
        w_recon=0.1,
        w_contrast=w_contrast,
        w_adv=w_adv,
        w_mmd=1.0,
        mmd_samples=256,
        grl_max=1.0,
        bimodal_min_batch_frac=0.5,
        ref_sample_per_marker=ref_sample_per_marker,
        **kw,
    )

    out_arm = mod.normalize_adata(
        adata, m_arm, sc_arm, ref_arm,
        hybrid_alpha=HYBRID_ALPHA,
        k_neighbors=K_NEIGHBORS,
        inference_batch_size=2048,
        device_str=DEVICE,
        layer_name='normalized',
        keep_base_layer=False,
        **kw,
    )

    adata_norm.layers[layer_key] = np.asarray(out_arm.layers['normalized'], dtype=np.float32)
    abl_history[layer_key] = h_arm
    print(f'  -> wrote adata_norm.layers[{layer_key!r}]  '
          f'final mmd={h_arm["mmd"][-1]:.5f}  loss={h_arm["loss"][-1]:.4f}')

    if layer_key == 'norm_fix':
        fix_model, fix_scaler = m_arm, sc_arm     # kept for the §4c sanity checks
    else:
        del m_arm
    del out_arm
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ---- training diagnostics --------------------------------------------------
# final_mmd  : batch alignment reached during training (lower = better).
# mean|delta|: how large a correction each arm applies (log1p space, vs Stage 1).
base_log = np.log1p(np.clip(np.asarray(adata_norm.layers['normalized_base']), 0, None))
diag_rows = []
for layer_key, label in ABL_LAYERS[1:]:
    h = abl_history[layer_key]
    arm_log = np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None))
    diag_rows.append({
        'arm': label,
        'layer': layer_key,
        'final_mmd': round(float(h['mmd'][-1]), 5),
        'final_loss': round(float(h['loss'][-1]), 4),
        'final_recon': round(float(h['recon'][-1]), 4),
        'final_contrast': round(float(h['contrast'][-1]), 4),
        'mean_abs_delta_log1p': round(float(np.abs(arm_log - base_log).mean()), 5),
    })
    del arm_log
del base_log

print('\n' + '=' * 70)
print('GRAPH-FIX A/B — training diagnostics (α=%.1f applied at inference)' % HYBRID_ALPHA)
print('=' * 70)
print(pd.DataFrame(diag_rows).to_string(index=False))
print('\nNOTE: final_contrast is not comparable across graphs — the fixed graph has many')
print('more NT-Xent positives per cell, so its contrastive loss is a different quantity.')
print(f'\nLayers now present: {list(adata_norm.layers.keys())}')

fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, key, title in zip(axes, ['mmd', 'recon', 'loss'],
                          ['MMD (decoder output)', 'Recon (Huber, = |delta|)', 'Total loss']):
    for layer_key, label in ABL_LAYERS[1:]:
        ax.plot(abl_history[layer_key][key], linewidth=1.6, label=label)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle('Graph-fix A/B — per-arm training curves', fontsize=13)
plt.tight_layout()
plt.show()

## 4d. External baseline — MMD-ResNet (Shaham et al., *Bioinformatics* 2017)

This is the closest prior method: a residual network trained to minimise MMD between batches,
published for mass cytometry. It is ported from the authors' Keras code (`mmd_resnet_baseline.py`).
The original's architecture, loss, kernel bandwidths, optimizer, learning-rate schedule and
early stopping are kept. It runs on **raw** data (no Stage 1) and on **all** markers (no bimodal
masking), as published.

**Adaptations.** The original maps one batch onto one other. Here there is one network per
non-reference batch, each mapping onto the reference batch. The reference batch is the one
holding the most KL-medoid reference samples, the same rule as CFM/DDPM. Training uses at most
50k cells per batch; each network is then applied to every cell of its batch.

The layer `norm_mmdresnet` joins every downstream section (§5–§8). §9 does **not** repeat it
over seeds.

In [ ]:
# ============================================================================
# 4d. MMD-ResNet baseline — one network per non-reference batch
# ============================================================================
mrn_nets, mrn_info = mrn.train_mmd_resnet(
    adata, ref_sample_per_marker,
    device_str=DEVICE,
    seed=ABL_SEED,
)
adata_norm.layers['norm_mmdresnet'] = mrn.apply_mmd_resnet(
    adata, mrn_nets, mrn_info, device_str=DEVICE)
if ('norm_mmdresnet', 'MMD-ResNet') not in ABL_LAYERS:
    ABL_LAYERS.append(('norm_mmdresnet', 'MMD-ResNet'))

print(f'Reference batch: {mrn_info["ref_batch"]}')
for b, h in mrn_info['history'].items():
    print(f'  {b}: {len(h["val"])} epochs, best val MMD {min(h["val"]):.5f}')
print(f'Layers evaluated downstream: {[lbl for _, lbl in ABL_LAYERS]}')

fig, ax = plt.subplots(figsize=(8, 4))
for b, h in mrn_info['history'].items():
    ax.plot(h['val'], linewidth=1.2, label=b)
ax.set_xlabel('Epoch')
ax.set_ylabel('validation MMD²')
ax.set_title('MMD-ResNet — validation MMD per source batch')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

del mrn_nets
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
gc.collect()

## 4c. Graph-fix sanity checks

These verify that the fixed graph code does what it claims, on the real data, using the
`graph-fix` model from §4b. Every check prints **PASS/FAIL**. Check 2 also shows how few
neighbours the **old** sampler actually gave each training cell.

In [ ]:
# ============================================================================
# 4c. GRAPH-FIX SANITY CHECKS  (uses fix_model / fix_scaler from §4b)
# ============================================================================
import time

assert fix_model is not None, 'run §4b first (it keeps the graph-fix model for these checks)'
dev = next(fix_model.parameters()).device
fix_model.eval()
results = {}

# [0] right model
mod_name = type(fix_model).__module__
results['0 model from spancy_shift_graphfix'] = mod_name == 'spancy_shift_graphfix'
print(f'[0] model class module: {mod_name}')

# shared graph objects (same construction as gf.train / gf.normalize_adata)
bc = adata.obs['batch_id'].astype('category').cat.codes.values.astype(np.int64)
scene_col = gf._get_col(adata, ['scene_id', 'sample_id'], 'batch_id')
sc = adata.obs[scene_col].astype('category').cat.codes.values.astype(np.int64)
xy = adata.obs[['x', 'y']].values.astype(np.float32)
sizes = np.bincount(sc)
print(f'    scenes ({scene_col}): {len(sizes)}  cells/scene median={int(np.median(sizes)):,}  '
      f'min={sizes.min():,}  max={sizes.max():,}')

knn = gf.build_knn_graphs(adata, k=K_NEIGHBORS)
adj = gf.AdjacencyIndex(knn)
rng = np.random.default_rng(0)

# [1] edge direction: each cell receives messages from exactly its own kNN
cells = rng.choice(adata.n_obs, 200, replace=False)
bad = 0
for i in cells:
    ctx = adj.expand(np.array([i]), hops=2)
    ei = adj.get_subgraph(ctx)
    got = set(ctx[ei[0, ei[1] == 0]].tolist())      # sources of edges into local node 0 = cell i
    want = set(knn[i].tolist()) - {int(i)}
    bad += got != want
results['1 edge direction (own kNN in)'] = bad == 0
print(f'[1] edge direction: {len(cells) - bad}/{len(cells)} cells receive exactly their own kNN')

# [2] training-graph density: old sampler vs patch sampler (+ halo)
def _mean_deg(sampler, n_steps=20):
    t, c = [], []
    for _ in range(n_steps):
        idx = sampler.sample()
        t.append(adj.target_in_degree(idx, adj.get_subgraph(idx)))
        c.append(adj.target_in_degree(idx, adj.get_subgraph(adj.expand(idx, hops=2))))
    return float(np.mean(t)), float(np.mean(c))

t_old, _ = _mean_deg(gf.SceneBasedSampler(bc, sc, n_per_batch=512))
t_new, c_new = _mean_deg(gf.PatchSampler(bc, sc, xy, n_per_batch=512, n_patches=4))
results['2 halo gives ~full kNN in training'] = c_new > 0.9 * K_NEIGHBORS
print(f'[2] mean neighbours present per training cell (full kNN = {K_NEIGHBORS}):')
print(f'    old sampler (what the old-graph arm trained on)  : {t_old:6.2f}')
print(f'    patch sampler, targets only (NT-Xent positives)  : {t_new:6.2f}')
print(f'    patch sampler + 2-hop halo (GAT input)           : {c_new:6.2f}')

# [3] NT-Xent: new memory-light version == old version (value and gradient)
z = torch.nn.functional.normalize(torch.randn(300, 32, dtype=torch.float64), dim=1)
e = torch.randint(0, 300, (2, 3000))
e = e[:, e[0] != e[1]]
z1 = z.clone().requires_grad_(True)
z2 = z.clone().requires_grad_(True)
l_old = spancy_shift.nt_xent_loss(z1, e, 0.07)
l_new = gf.nt_xent_loss(z2, e, 0.07)
g_old, = torch.autograd.grad(l_old, z1)
g_new, = torch.autograd.grad(l_new, z2)
ok3 = torch.allclose(l_old, l_new, atol=1e-8) and torch.allclose(g_old, g_new, atol=1e-8)
results['3 NT-Xent identical to old'] = bool(ok3)
print(f'[3] NT-Xent: old={l_old.item():.6f}  new={l_new.item():.6f}  '
      f'max|grad diff|={float((g_old - g_new).abs().max()):.1e}')

# [4]/[5] inference exactness and chunk-size invariance on one scene
Xs = torch.tensor(fix_scaler.transform(
    np.log1p(np.clip(np.asarray(adata_norm.layers['normalized_base']), 0, None))
).astype(np.float32))

@torch.no_grad()
def _delta(cells_, bs, halo=True):
    out = np.empty((len(cells_), Xs.shape[1]), np.float32)
    ratios = []
    for s in range(0, len(cells_), bs):
        ch = cells_[s:s + bs]
        ctx = adj.expand(ch, hops=2) if halo else ch
        ei = torch.tensor(adj.get_subgraph(ctx), dtype=torch.long, device=dev)
        _, d, _, _ = fix_model(Xs[ctx].to(dev), ei)
        out[s:s + len(ch)] = d[:len(ch)].cpu().numpy()
        ratios.append(len(ctx) / len(ch))
    return out, float(np.mean(ratios))

target = min(np.median(sizes), 30000)          # keep the whole-scene graph GPU-friendly
scene = int(np.argmin(np.abs(sizes - target)))
scene_cells = np.where(sc == scene)[0]
print(f'    test scene code {scene}: {len(scene_cells):,} cells')

whole, _ = _delta(scene_cells, len(scene_cells), halo=False)   # whole scene = one exact graph
timing = {}
halo_out = {}
for bs in (1024, 4096):
    t0 = time.time()
    halo_out[bs] = _delta(scene_cells, bs, halo=True)
    timing[bs] = time.time() - t0
d4 = float(np.abs(halo_out[1024][0] - whole).max())
d5 = float(np.abs(halo_out[1024][0] - halo_out[4096][0]).max())
nohalo_1k, _ = _delta(scene_cells, 1024, halo=False)
nohalo_4k, _ = _delta(scene_cells, 4096, halo=False)
d5_old = float(np.abs(nohalo_1k - nohalo_4k).max())
results['4 halo == whole-scene graph'] = d4 < 1e-4
results['5 chunk-size invariant'] = d5 < 1e-4
print(f'[4] max |delta(chunk+halo) - delta(whole scene)| = {d4:.2e}')
print(f'[5] max |delta(1024) - delta(4096)| with halo    = {d5:.2e}')
print(f'    contrast, WITHOUT halo (old-style chunking)  = {d5_old:.2e}  (non-zero = the old bug)')
for bs in (1024, 4096):
    print(f'    chunk={bs}: {timing[bs]:5.1f}s  context/chunk ratio={halo_out[bs][1]:.2f}  '
          '(~1.5-3 fine; >>10 = cells not spatially ordered, inference slow)')
del Xs, whole, halo_out, nohalo_1k, nohalo_4k

# [6] bimodal markers: decoder row all-zero -> graph-fix == Stage 1 exactly
names = list(adata.var_names)
X_raw_log = np.log1p(np.clip(np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X), 0, None))
is_bim, _ = detect_bimodal_markers(X_raw_log, names, batch_codes=bc)
del X_raw_log
last = fix_model.decoder.net[-1]
W, bvec = last.weight.detach().cpu().numpy(), last.bias.detach().cpu().numpy()
zero_rows = [m for m, w, b in zip(names, W, bvec) if not w.any() and b == 0]
bim_names = [m for m, b in zip(names, is_bim) if b]
j = [names.index(m) for m in bim_names]
gap = float(np.abs(np.log1p(np.asarray(adata_norm.layers['norm_fix'])[:, j]) -
                   np.log1p(np.asarray(adata_norm.layers['normalized_base'])[:, j])).max()) if j else 0.0
results['6 bimodal delta == 0'] = set(zero_rows) == set(bim_names) and gap < 1e-3
print(f'[6] bimodal markers: {bim_names}   zero decoder rows: {zero_rows}')
print(f'    max |log1p(graph-fix) - log1p(Stage 1)| on bimodal markers = {gap:.2e}')

# [7] every kNN row stays inside its own scene
rows = rng.choice(adata.n_obs, min(200_000, adata.n_obs), replace=False)
leak = int((sc[knn[rows]] != sc[rows][:, None]).any(axis=1).sum())
results['7 kNN stays inside scene'] = leak == 0
print(f'[7] kNN rows pointing outside their own scene: {leak} of {len(rows):,} sampled')

print('\n' + '=' * 60)
for name, ok in results.items():
    print(f'  {"PASS" if ok else "FAIL"}  {name}')
print('=' * 60)
print('ALL PASS' if all(results.values()) else 'SOME CHECKS FAILED — paste this output to Claude')
gc.collect()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels.
Lower = batch effect better removed.

In [ ]:
batch_arr = adata_norm.obs['batch_id'].values

# Falls back to the original two-layer view if §4b was not run.
ABL_LAYERS = globals().get('ABL_LAYERS',
                           [('normalized_base', 'Stage 1'), ('normalized', 'full')])

r2_compare = per_marker_batch_r2(
    np.log1p(np.clip(X_raw, 0, None)), batch_arr, marker_names
).rename(columns={'adj_r2': 'raw'})

for layer_key, label in ABL_LAYERS:
    r2_layer = per_marker_batch_r2(
        np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None)),
        batch_arr, marker_names,
    ).rename(columns={'adj_r2': label})
    r2_compare = r2_compare.merge(r2_layer, on='marker')

R2_COLS = ['raw'] + [label for _, label in ABL_LAYERS]

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in R2_COLS:
    print(f'  {col:>14s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(len(r2_compare))
n_bars = len(R2_COLS)
w = 0.8 / n_bars
palette_r2 = ['salmon', 'steelblue', 'forestgreen', 'darkorange', 'mediumorchid', 'sienna']

for i, col in enumerate(R2_COLS):
    offset = (i - (n_bars - 1) / 2) * w
    ax.bar(x + offset, r2_compare[col], w, label=col,
           color=palette_r2[i % len(palette_r2)], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better (raw + Stage 1 + ablation arms)')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using **per-sample GMM** (2-component Gaussian mixture
fitted on each sample's raw data in log10 space; threshold applied to both raw and normalized
for that sample). Delta = normalized − raw. Target: |delta| < 5% per marker.

**Headline metric**: the full per-marker mean Δ (± SD) table over all 20 markers, sorted by
mean Δ, with a `pass_5pct` column and a count of markers within ±5%. This is the primary
biology-preservation result. A high SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose
GMM threshold sits near the histogram peak — its large |Δ| is threshold instability, not real
biology distortion. Read mean Δ alongside its SD; do **not** drop markers.

The density-at-threshold reliability filter (`summarize_positive_population()`) is kept **for
reference only** at the end of the cell — it drops most unimodal markers (often leaving ~3),
too few to compare methods, so it is not the primary metric.

In [ ]:
ABL_LAYERS = globals().get('ABL_LAYERS',
                           [('normalized_base', 'Stage 1'), ('normalized', 'full')])

# Per-sample GMM positive-population table for every arm.
pop_tables_abl = {}
for layer_key, label in ABL_LAYERS:
    pop_tables_abl[label] = positive_population_table(
        adata_norm, norm_layer=layer_key, sample_col='sample_id'
    )

# Keep the original names so the heatmap / scatter / per-sample breakdown cells below
# keep working unchanged (they show the 'full' arm).
pop_table_base = pop_tables_abl['Stage 1']
pop_table = pop_tables_abl.get('graph-fix', pop_tables_abl[ABL_LAYERS[-1][1]])


# ── HEADLINE: full per-marker table, ALL 20 markers (primary biology metric) ──
# mean Δ = mean over samples of (norm − raw) % positive. SD = inter-sample spread.
# High SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose GMM threshold sits
# near the histogram peak → the large |Δ| reflects threshold instability, NOT real
# biology distortion. Read mean Δ alongside its SD; do not filter markers out.
def pop_summary(pt):
    s = pt.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw', 'mean'),
        mean_pct_norm=('pct_pos_norm', 'mean'),
        mean_delta=('delta', 'mean'),
        sd_delta=('delta', 'std'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
    ).round(2)
    s['pass_5pct'] = s['mean_delta'].abs() < 5
    return s.sort_values('mean_delta')


summaries = {label: pop_summary(pt) for label, pt in pop_tables_abl.items()}

# ---- cross-arm comparison (the ablation view) ------------------------------
delta_cmp = pd.DataFrame({label: s['mean_delta'] for label, s in summaries.items()})
sd_cmp = pd.DataFrame({label: s['sd_delta'] for label, s in summaries.items()})
delta_cmp = delta_cmp.reindex(summaries['Stage 1'].index)
sd_cmp = sd_cmp.reindex(delta_cmp.index)

print('=' * 95)
print('POSITIVE POPULATION — mean delta % per marker, per ablation arm (target |delta| < 5%)')
print('=' * 95)
print(delta_cmp.round(2).to_string())
print('\nInter-sample SD of delta (high SD = unimodal marker, threshold near the peak):')
print(sd_cmp.round(2).to_string())

print('\nMarkers within +/-5% mean delta:')
for label, s in summaries.items():
    print(f'  {label:>10s}: {int(s["pass_5pct"].sum())}/{len(s)}   '
          f'overall mean |delta| = {pop_tables_abl[label]["delta"].abs().mean():.2f}%')
print('\nIf the arms are within ~1pp of each other here, the dropped loss does not affect')
print('1D positive-population biology — decide on kBET / silhouette / shape instead.')

# ---- full per-arm tables ---------------------------------------------------
for label, s in summaries.items():
    print('\n' + '=' * 95)
    print(f'Positive Population Preservation — {label} vs raw  (ALL 20 markers):')
    print('=' * 95)
    print(s.to_string())
    print(f'\n  Markers within +/-5% mean delta: {int(s["pass_5pct"].sum())}/{len(s)}')

print('\nTarget: |mean delta| < 5% per marker. High-SD failing markers are unimodal (threshold')
print('near the histogram peak) — interpret their |delta| as threshold instability, not distortion.')

# ── Secondary (reference only): reliability-filtered view ─────────────────────
# The density-at-threshold filter drops most unimodal markers — often leaving only
# ~3 markers, too few to compare methods. Kept for reference; NOT the headline metric.
print('\n' + '─' * 95)
print('Secondary (reference only) — reliability-filtered; drops most markers, not primary:')
print('─' * 95)
for label, pt in pop_tables_abl.items():
    _ = summarize_positive_population(pt, min_reliable_frac=0.5, label=label)

In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: Normalized vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
plot_markers_pop = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names[:2]

fig, axes = plt.subplots(1, max(len(plot_markers_pop), 1),
                          figsize=(5 * max(len(plot_markers_pop), 1), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table[pop_table['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x (no change)')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (Stage 2 normalized)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation (Stage 2 vs Raw)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# === DETAILED PER-SAMPLE BREAKDOWN ===
# Show which samples are driving the large mean Δ values
# Identifies outlier samples and compares median vs mean

print('='*90)
print('DETAILED PER-SAMPLE BREAKDOWN — All Markers')
print('='*90)

S2_LABEL = 'Stage 2 GNN'
pop_tables = {'Stage 1': pop_table_base, S2_LABEL: pop_table}

for method_label, pt in pop_tables.items():
    print(f'\n{method_label} — Per-Sample Positive Population Delta\n')
    print('='*90)

    for marker in sorted(pt['marker'].unique()):
        marker_df = pt[pt['marker'] == marker].sort_values('delta')

        print(f'\n{marker}:')
        print('-'*90)
        print(marker_df[['sample', 'pct_pos_raw', 'pct_pos_norm', 'delta']].to_string(index=False))

        # Statistics
        mean_delta = marker_df['delta'].mean()
        median_delta = marker_df['delta'].median()
        std_delta = marker_df['delta'].std()
        min_delta = marker_df['delta'].min()
        max_delta = marker_df['delta'].max()

        within_5 = (abs(marker_df['delta']) <= 5).sum()
        total = len(marker_df)

        print(f'\nStats: Mean={mean_delta:+.2f}%, Median={median_delta:+.2f}%, '
              f'Std={std_delta:.2f}%, Range=[{min_delta:+.2f}%, {max_delta:+.2f}%]')
        print(f'Samples within ±5%: {within_5}/{total}')

        # Flag outliers
        outliers = marker_df[abs(marker_df['delta']) > 10]
        if len(outliers) > 0:
            print(f'⚠️  OUTLIERS (|Δ| > 10%):')
            for _, row in outliers.iterrows():
                print(f'    {row["sample"]}: {row["delta"]:+.2f}%')

    print(f'\n{method_label} SUMMARY:')
    print('-'*90)
    summary = pt.groupby('marker').agg({
        'delta': ['mean', 'median', 'std', 'min', 'max',
                  lambda x: (abs(x) <= 5).sum()]
    }).round(2)
    summary.columns = ['mean', 'median', 'std', 'min', 'max', 'within_5_count']
    print(summary.to_string())

    print(f'\nTarget: |mean Δ| < 5% per marker\n')

In [ ]:
# ============================================================================
# 6b. SHAPE-PRESERVATION DIAGNOSTIC  (location-invariant)  — per ablation arm
# ----------------------------------------------------------------------------
# Positive-pop Δ (fraction over a threshold) and silhouette (3-blob 20D
# separation) are both SHAPE-BLIND: you can flatten a sharp peak into a broad
# smear and still pass both. This cell measures whether each marker's 1D
# marginal SHAPE survives, INDEPENDENT of any location shift Stage 1 / GNN apply.
#
# Per (marker, sample), in log1p space, normalized vs raw (1.0 = shape preserved):
#   peak_ratio = peak density(norm) / peak density(raw)   <1 => mode flattened
#   var_ratio  = var(norm) / var(raw)                     >1 => broadened
#   iqr_ratio  = IQR(norm) / IQR(raw)                     >1 => broadened (robust)
#
# peak_ratio uses SHARED bin edges (union range) so it is binning-fair.
# Stage 1 is a pure translation -> should sit ~1.0 everywhere (control).
# NOTE: bimodal markers (ECAD etc.) are masked from the GNN's MMD loss, so their
# delta stays ~0 and they trivially score ~1.0 for every arm — they prove nothing
# about the ablation. Judge the arms on the unimodal markers the GNN corrects.
#
# ABLATION READING: if an arm drops a loss and peak/var stay ~1.0 while kBET holds,
# that loss was not buying shape safety.
# ============================================================================
from scipy.stats import iqr as _iqr

SHAPE_LAYERS = globals().get('ABL_LAYERS',
                             [('normalized_base', 'Stage 1'), ('normalized', 'full')])
SHAPE_LABELS = [label for _, label in SHAPE_LAYERS]
S1_LABEL = SHAPE_LABELS[0]                                  # control (pure shift)
MAIN_LABEL = SHAPE_LABELS[1] if len(SHAPE_LABELS) > 1 else S1_LABEL
N_SHAPE_BINS = 80
MIN_CELLS    = 50

# --- precompute ONCE ---
sample_ids_shape = adata_norm.obs['sample_id'].values
sample_unique    = np.unique(sample_ids_shape)
sample_idx       = {s: np.where(sample_ids_shape == s)[0] for s in sample_unique}
sample_idx       = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log        = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


rows = []
for layer_key, layer_label in SHAPE_LAYERS:
    X_layer_log = np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        raw_col  = X_raw_log[:, mi]
        norm_col = X_layer_log[:, mi]
        peaks, vrs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(raw_col[idx], norm_col[idx])
            if np.isfinite(p):  peaks.append(p)
            if np.isfinite(v):  vrs.append(v)
            if np.isfinite(ir): irs.append(ir)
        rows.append({
            'method': layer_label, 'marker': mname,
            'peak_ratio': np.mean(peaks) if peaks else np.nan,
            'var_ratio':  np.mean(vrs)  if vrs  else np.nan,
            'iqr_ratio':  np.mean(irs)  if irs  else np.nan,
        })
    del X_layer_log
shape_df = pd.DataFrame(rows)

# ---- side-by-side table: every arm per marker ------------------------------
piv = shape_df.pivot(index='marker', columns='method',
                     values=['peak_ratio', 'var_ratio', 'iqr_ratio']).round(3)
piv = piv.reindex(columns=pd.MultiIndex.from_product(
    [['peak_ratio', 'var_ratio', 'iqr_ratio'], SHAPE_LABELS]))
piv = piv.sort_values(('peak_ratio', MAIN_LABEL))

print('=' * 92)
print('SHAPE-PRESERVATION DIAGNOSTIC  (1.0 = shape preserved; per-sample mean vs raw, log1p)')
print('  peak_ratio <1 = mode flattened   |   var_ratio / iqr_ratio >1 = broadened')
print('=' * 92)
print(piv.to_string())

by_method = {label: shape_df[shape_df.method == label].set_index('marker')
             for label in SHAPE_LABELS}
s1 = by_method[S1_LABEL]
PEAK_LO, VAR_HI, VAR_LO = 0.80, 1.25, 0.80

# ---- per-arm distortion flags ----------------------------------------------
print('\n' + '=' * 92)
print(f'SHAPE-DISTORTED markers per arm (peak<{PEAK_LO} or var outside [{VAR_LO},{VAR_HI}])')
print('=' * 92)
for label in SHAPE_LABELS[1:]:
    arm = by_method[label]
    distorted = arm[(arm.peak_ratio < PEAK_LO) |
                    (arm.var_ratio > VAR_HI) | (arm.var_ratio < VAR_LO)]
    print(f'\n{label}:  {len(distorted)} distorted')
    if len(distorted) == 0:
        print('  none — preserves 1D shape on all markers')
    else:
        for m, r in distorted.sort_values('peak_ratio').iterrows():
            tag = ' [bimodal/passed-through]' if m in bimodal_names else ''
            print(f'  {m:>10s}  peak={r.peak_ratio:.2f} (S1 {s1.loc[m,"peak_ratio"]:.2f})  '
                  f'var={r.var_ratio:.2f} (S1 {s1.loc[m,"var_ratio"]:.2f})  '
                  f'iqr={r.iqr_ratio:.2f}{tag}')

# ---- unimodal-only summary across arms -------------------------------------
uni = [m for m in s1.index if m not in bimodal_names]
uni_rows = []
for label in SHAPE_LABELS:
    arm = by_method[label]
    flagged = arm.loc[uni]
    uni_rows.append({
        'arm': label,
        'mean_peak_ratio': round(float(flagged['peak_ratio'].mean()), 3),
        'mean_var_ratio':  round(float(flagged['var_ratio'].mean()), 3),
        'mean_iqr_ratio':  round(float(flagged['iqr_ratio'].mean()), 3),
        'n_distorted': int(((flagged.peak_ratio < PEAK_LO) |
                            (flagged.var_ratio > VAR_HI) |
                            (flagged.var_ratio < VAR_LO)).sum()),
    })
print('\n' + '=' * 92)
print('Unimodal markers only (the ones the GNN actually corrects) — 1.0 = shape preserved')
print('=' * 92)
print(pd.DataFrame(uni_rows).to_string(index=False))
print(f'\n{S1_LABEL} is the control: a pure translation, so it should sit at ~1.0.')
print('An arm that matches it has not bought its kBET by reshaping marginals.')

# ---- bar charts: peak_ratio and var_ratio across arms ----------------------
order = list(piv.index)
x = np.arange(len(order))
n_m = len(SHAPE_LABELS)
w = 0.8 / n_m
palette_shape = ['steelblue', 'forestgreen', 'darkorange', 'mediumorchid', 'sienna']
fig, axes = plt.subplots(2, 1, figsize=(20, 11))
for ax, ratio, ref_lines in [
    (axes[0], 'peak_ratio', [1.0]),
    (axes[1], 'var_ratio',  [1.0, VAR_HI, VAR_LO]),
]:
    for i, label in enumerate(SHAPE_LABELS):
        offset = (i - (n_m - 1) / 2) * w
        ax.bar(x + offset, [by_method[label].loc[m, ratio] for m in order], w,
               label=label, color=palette_shape[i % len(palette_shape)], alpha=0.85)
    for rl in ref_lines:
        ax.axhline(rl, color='red', linestyle='--', alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{ratio}  (1.0 = preserved)')
    ax.set_ylabel(ratio)
    ax.legend(fontsize=8, ncol=n_m)
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Shape preservation per ablation arm (Stage 1 = pure-shift control)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nReading it: Stage 1 bars hug 1.0 (translation preserves shape). Every arm should also')
print('stay near 1.0 — if a reduced arm departs, the dropped loss WAS protecting 1D shape.')

## 7. Histogram Comparison

Per-sample histogram PDFs for all markers. Saved to `histograms_shift/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_shift'
os.makedirs(save_dir, exist_ok=True)

# Raw is in .X (not a layer) -> use a sentinel key and fetch it specially.
# One column per ablation arm, so you can SEE what dropping a loss does to the
# per-sample histograms (the shape table in 6b is the numeric version of this).
RAW = '__RAW__'
_abl = globals().get('ABL_LAYERS',
                     [('normalized_base', 'Stage 1 (analytic)'), ('normalized', 'Stage 2')])
layers_to_plot = [(RAW, 'Raw')] + list(_abl)

def _get_X(key):
    if key == RAW:
        Xr = adata_norm.X
        return Xr.toarray() if hasattr(Xr, 'toarray') else np.asarray(Xr)
    return np.asarray(adata_norm.layers[key])

sample_col = 'sample_id' if 'sample_id' in adata_norm.obs.columns else 'batch_id'
sample_ids = adata_norm.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'shift_histograms_ablation.pdf')
n_cols = len(layers_to_plot)
rows_per_page = 3 if n_cols > 4 else 4   # keep pages readable once arms are added

# CROSS-PDF MATCH: derive bin edges AND y-limit per marker from RAW ONLY.
# Raw is identical in this notebook and in mxnorm_benchmark, so both PDFs get
# the SAME x-bins and SAME y-axis per marker -> directly comparable side by side.
# Here it also means every ARM shares one scale, so columns are comparable.
def _raw_log(m_idx):
    col = np.asarray(_get_X(RAW)[:, m_idx]).ravel()
    return np.log1p(np.clip(col, 0, None))

def _marker_edges(m_idx, n_bins=80):
    col = _raw_log(m_idx)
    lo, hi = float(col.min()), float(col.max())
    if not np.isfinite(lo) or hi <= lo:
        hi = lo + 1.0
    return np.linspace(lo, hi, n_bins + 1)

def _marker_ymax(m_idx, edges):
    X_log = _raw_log(m_idx)
    ymax = 0
    for s in unique_samples:
        c, _ = np.histogram(X_log[sample_ids == s], bins=edges)
        ymax = max(ymax, int(c.max()) if c.size else 0)
    return max(ymax, 1)

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), n_cols,
                                      figsize=(5.5 * n_cols, 4 * len(page_markers)))
            axes = np.atleast_2d(axes)
            if axes.shape != (len(page_markers), n_cols):
                axes = axes.reshape(len(page_markers), n_cols)

            for row, mname in enumerate(page_markers):
                m_idx   = marker_names.index(mname)
                edges   = _marker_edges(m_idx)            # from RAW
                centers = 0.5 * (edges[:-1] + edges[1:])
                ymax    = _marker_ymax(m_idx, edges) * 1.05   # from RAW
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_log = np.log1p(np.clip(np.asarray(_get_X(layer_key)[:, m_idx]).ravel(), 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, _ = np.histogram(X_log[mask], bins=edges)   # SHARED edges
                        ax.plot(centers, c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    ax.set_ylim(0, ymax)                   # SHARED y across all columns
                    ax.set_xlim(edges[0], edges[-1])
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')
print(f'Columns: {[label for _, label in layers_to_plot]}')
print('Bins + y-axis per marker derived from RAW only, so all arms share identical axes')
print('and can be compared column-to-column (and against mxnorm_histograms.pdf).')

In [ ]:
# 7b. Export GNN Stage 2 + Stage 1 layers for the cross-method histogram figure
# (thesis_histograms.ipynb: Raw | Stage 1 | GNN | CFM | DDPM).
# No Google Drive access: files are written to local runtime disk and downloaded to
# your computer — upload them into thesis_histograms at /content/spancy_layers/.
# Stage 1 is identical in all three notebooks (shared shift_normalize_per_marker),
# so it is exported once, from here. obs_names saved to verify cell order.
import os, json
from google.colab import files

EXPORT_DIR = '/content/spancy_layers'
os.makedirs(EXPORT_DIR, exist_ok=True)

for key, layer, label in [
      ('gnn',    'normalized',      f'Stage 2 GNN (α={HYBRID_ALPHA})'),
      ('stage1', 'normalized_base', 'Stage 1 (analytic)'),
]:
      np.save(f'{EXPORT_DIR}/{key}_normalized.npy',
              np.asarray(adata_norm.layers[layer], dtype=np.float32))
      np.save(f'{EXPORT_DIR}/{key}_obs_names.npy',
              adata_norm.obs_names.astype(str).to_numpy())
      with open(f'{EXPORT_DIR}/{key}_meta.json', 'w') as f:
          json.dump({'method': key, 'label': label, 'n_epochs': N_EPOCHS,
                     'shape': list(adata_norm.shape)}, f, indent=2)
      print(f'Saved {key} ({layer}) {adata_norm.shape} -> {EXPORT_DIR}')
      for suffix in ('normalized.npy', 'obs_names.npy', 'meta.json'):
          files.download(f'{EXPORT_DIR}/{key}_{suffix}')

## 8. kBET Evaluation

Compute kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- **Lower chi-square = better**

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

# NOTE: the original notebook materialised adata_norm.layers['raw'] here, but nothing
# reads it — the kBET loop iterates EVAL_LAYERS (raw is not one) and the silhouette cell
# special-cases 'raw' by reading the X_raw array directly. Skipped to save ~141 MB now
# that five arm layers are resident.

# One entry per ablation arm (Stage 1 + full + the reduced arms).
ABL_LAYERS = globals().get('ABL_LAYERS',
                           [('normalized_base', 'Stage 1'), ('normalized', 'full')])
EVAL_LAYERS = [layer_key for layer_key, _ in ABL_LAYERS]
ABL_LABEL = {layer_key: label for layer_key, label in ABL_LAYERS}

print(f'Evaluating {len(EVAL_LAYERS)} layers: {EVAL_LAYERS}')
print(f'Arm labels: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')
print()
print('NOTE: kBET runs a UMAP per (layer x group), so this is ~5x the usual cost.')
print()
print('Baselines (g1–g5 mean kBET):')
print('  Raw (no correction):                ~0.176')
print('  shift_normalize.py (Stage 1 scipy): 0.6307')
print('  UniFORM:                            0.631')
print('  Stage 2 GNN full loss (alpha=0.6):  ~0.708   <- the number the arms must match')
print()
print('Ablation question: does dropping NT-Xent (no-NTX) or the adversarial CE (no-CE)')
print('move kBET away from the full-loss arm by more than seed noise?')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {
            'arm': ABL_LABEL.get(layer_name, layer_name),
            'layer': layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'n_groups': f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {row['arm']:>10s} ({layer_name:>15s})  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")

df_kbet = pd.DataFrame(summary_rows)

# ---- per-group breakdown, all arms side by side ----------------------------
print('\n' + '=' * 70)
print('Per-group kBET (g3/g4/g5 are the hard cross-batch pairs)')
print('=' * 70)
pergroup = pd.DataFrame({
    ABL_LABEL.get(layer_name, layer_name): {g: v['kBET'] for g, v in res.items()}
    for layer_name, res in all_kbet.items()
}).round(4)
print(pergroup.to_string())

# ---- the ablation verdict --------------------------------------------------
if 'old-graph' in df_kbet['arm'].values:
    full_k = float(df_kbet.loc[df_kbet['arm'] == 'old-graph', 'mean_kBET'].iloc[0])
    print('\n' + '=' * 70)
    print(f'ABLATION VERDICT — vs old-graph (kBET {full_k:.4f})')
    print('=' * 70)
    for _, r in df_kbet.iterrows():
        if r['arm'] in ('old-graph', 'Stage 1'):
            continue
        d = r['mean_kBET'] - full_k
        verdict = ('BETTER than old-graph' if d > 0.02 else
                   'WORSE than old-graph'  if d < -0.02 else
                   'within +/-0.02 of full (likely noise — needs a seed repeat)')
        print(f"  {r['arm']:>10s}: {r['mean_kBET']:.4f}  ({d:+.4f})  -> {verdict}")
    print('\nA reduced arm that lands within noise of `full` means the dropped loss is not')
    print('earning its place. Confirm with ABL_SEED = 1 and 2 before concluding.')

print()
print('Baselines:')
print('  Stage 1 expected:            kBET~0.620  (analytic, matches scipy shift_normalize.py)')
print('  UniFORM:                     kBET=0.631')
print(f'  Full-loss GNN (alpha={HYBRID_ALPHA}):  kBET~0.708')

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    palette_k = ['steelblue', 'forestgreen', 'darkorange', 'mediumorchid', 'sienna']
    bar_colors = [palette_k[i % len(palette_k)] for i in range(len(df_kbet))]

    axes[0].bar(df_kbet['arm'], df_kbet['mean_kBET'], color=bar_colors, alpha=0.85)
    axes[0].axhline(0.631, color='red', linestyle='--', linewidth=1.5, label='UniFORM 0.631')
    if 'old-graph' in df_kbet['arm'].values:
        axes[0].axhline(full_k, color='black', linestyle=':', linewidth=1.5, label='old-graph')
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET per ablation arm (higher = better)')
    axes[0].set_xticklabels(df_kbet['arm'], rotation=20, ha='right', fontsize=9)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].bar(df_kbet['arm'], df_kbet['mean_chi2'], color=bar_colors, alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['arm'], rotation=20, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
# Silhouette score — UniFORM-style: 3 cell types, computed PER SAMPLE
# Tumor epithelial (ECAD+), immune (CD45+), non-immune stromal (aSMA+)
#
# Why per-sample (not per clinical group): computing silhouette across two
# batch-paired samples lets the cross-batch offset inflate the raw baseline —
# good normalization then looks like it *lowers* silhouette (it just removed the
# batch-driven separation). Computing within each sample removes that confound:
# the score reflects only whether the 3 cell populations stay distinct.
# Labels are assigned once on raw data, fixed across all methods.

N_SIL = 3000           # cells per sample; O(n²) — stable well below this
MIN_TYPE_CELLS = 50    # a cell type counts as "present" if it has >= this many cells
                        # (decided on FULL per-sample counts → deterministic, seed-independent)

# Step 1: GMM thresholds for 3 anchor markers on full raw dataset (log1p space)
anchor_names = ['ECAD', 'CD45', 'aSMA']
anchor_thresholds = {}
print('Anchor marker GMM thresholds (log1p, raw data):')
for mname in anchor_names:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(
        X_log_preview[:, mi].reshape(-1, 1)
    )
    thr = float(np.mean(gmm.means_))
    anchor_thresholds[mname] = thr
    print(f'  {mname}: {thr:.3f}')

# Step 2: assign cell type labels on raw data — fixed for all methods
#   0 = tumor epithelial (ECAD+)
#   1 = immune           (CD45+)
#   2 = stromal          (aSMA+)
#  -1 = unassigned
# Priority for conflicts: ECAD > CD45 > aSMA
ecad_pos = X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']
cd45_pos = X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']
asma_pos = X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[asma_pos] = 2
cell_type[cd45_pos] = 1
cell_type[ecad_pos] = 0

print('\nCell type assignments (raw data):')
for code, label in [(0, 'tumor epithelial (ECAD+)'),
                    (1, 'immune (CD45+)'),
                    (2, 'stromal (aSMA+)'),
                    (-1, 'unassigned')]:
    n = (cell_type == code).sum()
    print(f'  {label}: {n:,}  ({100*n/len(cell_type):.1f}%)')

# Step 3: silhouette per sample, per method (no cross-batch mixing)
# Guard (deterministic): a sample is kept iff >= 2 cell types each have
# >= MIN_TYPE_CELLS cells on FULL data. Only those well-populated types are
# carried into the silhouette; sparse types (e.g. near-absent immune) are
# excluded from the computation rather than letting a subsample lottery drop
# the whole sample. This makes membership seed-independent and reproducible
# across notebooks.
print(f'\nSilhouette score per sample — 3 cell types in 20D log1p space (higher = better):')
sample_ids_all = adata_norm.obs['sample_id'].values
sil_rows = []
dropped = []
for sample in sorted(np.unique(sample_ids_all)):
    smask = sample_ids_all == sample
    s_labels = cell_type[smask]

    # full-data per-type counts → which types are substantial
    full_counts = {c: int((s_labels == c).sum()) for c in (0, 1, 2)}
    big = [c for c, n in full_counts.items() if n >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue  # need >= 2 substantial cell types to define a silhouette

    # keep only the substantial types, then subsample for the O(n²) silhouette
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sel_labels = s_labels[sel]

    sil_by_layer = {}
    for layer_name in ['raw'] + EVAL_LAYERS:
        if layer_name == 'raw':
            Xs = X_raw[smask]
        else:
            Xs = np.asarray(adata_norm.layers[layer_name])[smask]
        X_log = np.log1p(np.clip(Xs[sel], 0, None))
        sil_by_layer[layer_name] = round(
            float(silhouette_score(X_log, sel_labels, metric='euclidean')), 4
        )
    sil_rows.append({'sample': sample, **sil_by_layer})

sil_df = pd.DataFrame(sil_rows).set_index('sample')
sil_df.loc['Mean'] = sil_df.mean().round(4)
print(sil_df.to_string())
print(f'\n({len(sil_rows)} samples passed the guard: >=2 cell types with >={MIN_TYPE_CELLS} cells)')
if dropped:
    print(f'(dropped {len(dropped)}: {", ".join(dropped)})')
print('(higher = better — tumor / immune / stromal populations stay distinct after normalization)')


In [ ]:
VIZ_LAYER = 'normalized'
palette = plt.cm.tab10.colors
fig, axes = plt.subplots(5, 2, figsize=(12, 28))
fig.suptitle('Batch mixing per clinical group — Raw vs Normalized', fontsize=14, y=1.01)

for row_idx, (gname, pairs) in enumerate(GROUP_DEFS.items()):
    parts = [subset_ab(adata_norm, s, b) for s, b in pairs]
    adata_g = parts[0].concatenate(parts[1], batch_key=None)
    unique_b = sorted(adata_g.obs['batch_id'].unique())

    # Raw UMAP — compute via pegasus (same pipeline as kBET)
    umap_raw, batch_raw = None, None
    try:
        mmdata_raw = pgio.MultimodalData(adata_g.copy())
        pg.qc_metrics(mmdata_raw); pg.filter_data(mmdata_raw)
        pg.identify_robust_genes(mmdata_raw)
        pg.pca(mmdata_raw, features=None, n_components=20)
        pg.neighbors(mmdata_raw); pg.umap(mmdata_raw)
        umap_raw  = mmdata_raw.obsm['X_umap']
        batch_raw = mmdata_raw.obs['batch_id'].values
    except Exception as e:
        print(f'{gname} raw UMAP failed: {e}')

    # Normalized UMAP — reuse from kBET loop (no recomputation)
    umap_nrm, batch_nrm = None, None
    if VIZ_LAYER in umap_coords and gname in umap_coords[VIZ_LAYER]:
        umap_nrm, batch_nrm = umap_coords[VIZ_LAYER][gname]

    for col_idx, (coords, batch_ids, title) in enumerate([
            (umap_raw, batch_raw, 'Raw'),
            (umap_nrm, batch_nrm, f'Normalized ({VIZ_LAYER})'),
    ]):
        ax = axes[row_idx, col_idx]
        if coords is None:
            ax.text(0.5, 0.5, 'UMAP unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{gname} — {title}', fontsize=10)
            continue
        for ci, b in enumerate(unique_b):
            mask = batch_ids == b
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=[palette[ci % len(palette)]], s=2, alpha=0.3, rasterized=True, label=b)
        ax.set_title(f'{gname} — {title}', fontsize=10)
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)
        if col_idx == 1:
            ax.legend(markerscale=4, fontsize=8, loc='upper right',
                      title='batch_id', title_fontsize=8)

plt.tight_layout()
plt.savefig('umap_batch_mixing.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: umap_batch_mixing.png')

## 9. Seed repeat — are the differences larger than run-to-run noise?

§4b/§8 ran **one seed per arm**. The same old-graph configuration gave 0.7275 here but 0.7431 in the
loss ablation, so run-to-run noise may be about 0.016 kBET. This section retrains **old-graph**,
**graph-fix** and **no-graph** (all `w_contrast=0.5`, `w_adv=0.0`) at seeds 1 and 2. For each arm
it reports the **paired difference versus old-graph** within each seed. Seed 0 is reused from §8.

**Read the paired columns, not the means.** Each comparison gets one of three verdicts:
- consistently better;
- consistently worse;
- inconsistent (within noise).

For **no-graph**, "within noise" is the expected and useful outcome: it means the graph never mattered.

**Cost:** 6 trainings plus 30 kBET/UMAP group runs. Each layer is evaluated and then discarded, so
this adds no resident memory.

In [ ]:
# ============================================================================
# 9. SEED REPEAT — old-graph vs graph-fix vs no-graph at seeds 1 and 2 (seed 0 from §8)
# ----------------------------------------------------------------------------
# Requires section 8 to have run (pg, pgio, GROUP_DEFS, subset_ab, all_kbet).
# Each arm is trained, evaluated, and DISCARDED — nothing is added to adata_norm.
# ============================================================================

SEED_REPEAT = [1, 2]
# label: (module, extra kwargs)  — all w_contrast=0.5, w_adv=0.0
REPEAT_ARMS = {
    'old-graph': (spancy_shift, {}),
    'graph-fix': (gf, {}),
    'no-graph':  (gf, {'graph_mode': 'none'}),
}
CONTROL = 'old-graph'


def _kbet_mean(X_arr):
    """Mean kBET over the 5 clinical groups for one expression matrix."""
    a = ad.AnnData(
        X=np.asarray(X_arr, dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )
    scores = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(a, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            _stat, _p, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            scores[gname] = float(score)
        except Exception as e:
            print(f'    {gname}: FAILED — {e}')
            scores[gname] = float('nan')
        finally:
            parts = adata_g = mmdata = None
    del a
    gc.collect()
    vals = [v for v in scores.values() if not np.isnan(v)]
    return (float(np.mean(vals)) if vals else float('nan')), scores


# ---- seed 0 results, reused from the main run (no retraining) --------------
repeat_rows = []
for layer_key, arm_label in [('normalized', 'old-graph'), ('norm_fix', 'graph-fix'),
                             ('norm_nograph', 'no-graph')]:
    if layer_key in all_kbet:
        per_g = {g: v['kBET'] for g, v in all_kbet[layer_key].items()}
        vals = [v for v in per_g.values() if not np.isnan(v)]
        repeat_rows.append({'seed': ABL_SEED, 'arm': arm_label,
                            'kBET': float(np.mean(vals)), **{f'kBET_{g}': v for g, v in per_g.items()}})
        print(f'seed {ABL_SEED}  {arm_label:>9s}: kBET={np.mean(vals):.4f}  (reused from section 8)')

# ---- retrain at the extra seeds -------------------------------------------
for seed in SEED_REPEAT:
    for arm_label, (mod, kw) in REPEAT_ARMS.items():
        print('\n' + '=' * 70)
        print(f'SEED {seed}  ARM {arm_label}   (module={mod.__name__}  {kw})')
        print('=' * 70)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        m_s, sc_s, ref_s, _h_s = mod.train(
            adata,
            n_epochs=N_EPOCHS,
            lr=1e-3,
            device_str=DEVICE,
            warmup_epochs=2,
            n_per_batch=512,
            k_neighbors=K_NEIGHBORS,
            hybrid_alpha=HYBRID_ALPHA,
            temperature=0.07,
            w_recon=0.1,
            w_contrast=0.5,
            w_adv=0.0,
            w_mmd=1.0,
            mmd_samples=256,
            grl_max=1.0,
            bimodal_min_batch_frac=0.5,
            ref_sample_per_marker=ref_sample_per_marker,
            **kw,
        )
        out_s = mod.normalize_adata(
            adata, m_s, sc_s, ref_s,
            hybrid_alpha=HYBRID_ALPHA,
            k_neighbors=K_NEIGHBORS,
            inference_batch_size=2048,
            device_str=DEVICE,
            layer_name='normalized',
            keep_base_layer=False,
            **kw,
        )

        mean_k, per_g = _kbet_mean(out_s.layers['normalized'])
        repeat_rows.append({'seed': seed, 'arm': arm_label, 'kBET': mean_k,
                            **{f'kBET_{g}': v for g, v in per_g.items()}})
        print(f'  -> seed {seed} {arm_label}: mean kBET = {mean_k:.4f}')

        del m_s, out_s
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

# ---- verdict ---------------------------------------------------------------
rep_df = pd.DataFrame(repeat_rows)
print('\n' + '=' * 78)
print('SEED REPEAT — kBET per seed and arm')
print('=' * 78)
print(rep_df.round(4).to_string(index=False))

wide = rep_df.pivot(index='seed', columns='arm', values='kBET')
others = [a for a in REPEAT_ARMS if a != CONTROL and a in wide.columns]
if CONTROL in wide.columns and others:
    for arm in others:
        wide[f'{arm} minus {CONTROL}'] = wide[arm] - wide[CONTROL]
    print('\n' + '=' * 78)
    print('PAIRED DIFFERENCES within each seed (this is the statistic that matters)')
    print('=' * 78)
    print(wide.round(4).to_string())

    print(f'\nper-arm mean +/- SD across seeds:')
    for arm in [CONTROL] + others:
        col = wide[arm].dropna()
        print(f'  {arm:>9s}: {col.mean():.4f} +/- {col.std():.4f}  (n={len(col)})')

    for arm in others:
        d = wide[f'{arm} minus {CONTROL}'].dropna()
        print('\n' + '-' * 78)
        print(f'{arm} minus {CONTROL}: mean {d.mean():+.4f}  SD {d.std():.4f}  '
              f'range [{d.min():+.4f}, {d.max():+.4f}]  n={len(d)}')
        big = abs(d.mean()) > 3 * max(d.std(), 1e-6)
        if len(d) >= 2 and (d > 0).all():
            print(f'VERDICT: {arm} beats {CONTROL} at EVERY seed'
                  + (' and the effect is large relative to its spread.' if big else
                     ', but the spread is not negligible — report mean +/- SD.'))
        elif len(d) >= 2 and (d < 0).all():
            print(f'VERDICT: {arm} is WORSE than {CONTROL} at every seed.')
        else:
            print(f'VERDICT: sign NOT consistent across seeds -> {arm} vs {CONTROL} is within noise.')
        if arm == 'graph-fix':
            print('  (graph-fix also compressed 1D shape in §6 — a kBET tie is not a win.)')
        if arm == 'no-graph':
            print('  (no-graph within noise = the graph never mattered: the published model is a')
            print('   per-cell residual corrector. Compare §6 shape / pos-pop for no-graph too.)')